# 04. Real-Simulation Extractor Parity

此处的 parity 不要求真实 EEG 与模型 output 分布完全相同；它要求 input signal、feature name/order、unit、aggregation、validity semantics 和 vector shape 可被同一 observation contract 解释。

**硬边界：** cortical/thalamic firing rate 不是 simulated EEG。将 rate 输入 EEG extractor，或将旧 5D/7D banks 重命名为 14D/23D，都不是 parity。

In [1]:
from __future__ import annotations
import json
import os
from pathlib import Path
import sys
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / ".git").exists() and (PROJECT_ROOT / "S4_sbi").exists()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate repository root")
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "S4_sbi" / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from sleep_sbi import overnight_ablation as oa

print("project root:", PROJECT_ROOT)
print("CONDA_DEFAULT_ENV:", os.environ.get("CONDA_DEFAULT_ENV"))
print("sys.executable:", sys.executable)
print("sys.prefix:", sys.prefix)
print("Python:", sys.version)
print("workflow version:", oa.SCHEMA_VERSION)
assert os.environ.get("CONDA_DEFAULT_ENV") == "neurolib"
assert "neurolib" in sys.executable.lower()
assert "neurolib" in sys.prefix.lower()

project root: D:\Year3_Mao_Projects\sleep_loop
CONDA_DEFAULT_ENV: neurolib
sys.executable: C:\Users\YUS190\AppData\Local\anaconda3\envs\neurolib\python.exe
sys.prefix: C:\Users\YUS190\AppData\Local\anaconda3\envs\neurolib
Python: 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:36:49) [MSC v.1944 64 bit (AMD64)]
workflow version: overnight-observation-ablation-v0.1


## Audit existing simulator assets

只读取已有 V7/V8/V8a candidates、old simulation banks、raw rate diagnostic 和 simulator source evidence；不运行大规模 simulation。

In [2]:
parity_result = oa.run_extractor_parity()
display(parity_result["banks"])
display(oa.concise_status_frame(parity_result["parity"], ["schema", "dimension", "real_shape", "simulation_shape", "order_match", "fixed_length_simulation", "same_semantics", "parity_pass", "gate"]))
print("Parity artifact directory:", parity_result["output_dir"])

,path,keys,theta_shape,observation_shape,summary_keys,finite_observation_values,has_object_arrays,schema_match_14d,schema_match_23d
0,S4_sbi/sbi_outputs/all_simulations.npz,"[theta, x, param_names, summary_keys]","[5000, 4]","[5000, 5]","[shape_r, T4_q, T4_freq, T8_n_sp_events, T11_l...",True,False,False,False
1,S4_sbi/sbi_outputs_7dim_archive_20260507/all_s...,"[theta, x, param_names, summary_keys]","[4000, 4]","[4000, 7]","[shape_r, T4_q, T4_freq, T6_ibi_cv, T8_n_sp_ev...",True,False,False,False


,schema,dimension,real_shape,simulation_shape,order_match,fixed_length_simulation,same_semantics,parity_pass,gate
0,A_baseline14,14,"(142, 14)","(5000, 5)",False,False,False,False,NO-GO
1,B_baseline_plus_so_morphology,16,"(142, 16)","(5000, 5)",False,False,False,False,NO-GO
2,C_baseline_plus_spindle_occupancy,15,"(142, 15)","(5000, 5)",False,False,False,False,NO-GO
3,D_baseline_plus_event_phase,17,"(142, 17)","(5000, 5)",False,False,False,False,NO-GO
4,E_recommended_frozen_augmented,not frozen,not applicable,"(5000, 5)",False,False,False,False,NO-GO
5,F_complete23_diagnostic,23,"(142, 23)","(5000, 5)",False,False,False,False,NO-GO


Parity artifact directory: D:\Year3_Mao_Projects\sleep_loop\S4_sbi\results\overnight_observation_ablation\extractor_parity


## Feature-level failure evidence

每行 failure 是一个可恢复工程/科学门槛，不是缺失 result。current simulator outputs 的 legacy rate-summary schema 必须和 real EEG schema 分开记录。

In [3]:
display(parity_result["failures"].head(40))
display(pd.read_csv(parity_result["output_dir"] / "source_parity_evidence.csv"))
display(pd.read_csv(parity_result["output_dir"] / "candidate_asset_inventory.csv"))
display(pd.read_csv(parity_result["output_dir"] / "representative_parameter_assets.csv"))

,schema,feature,failure_category,reason,severity
0,A_baseline14,fooof_aperiodic_exponent,missing_simulated_EEG_observable_contract,"No same-unit, same-channel, same-preprocessing...",hard_gate
1,A_baseline14,so_peak_frequency_hz,missing_simulated_EEG_observable_contract,"No same-unit, same-channel, same-preprocessing...",hard_gate
2,A_baseline14,relative_so_power,missing_simulated_EEG_observable_contract,"No same-unit, same-channel, same-preprocessing...",hard_gate
3,A_baseline14,so_q,missing_simulated_EEG_observable_contract,"No same-unit, same-channel, same-preprocessing...",hard_gate
4,A_baseline14,so_event_rate_per_min,missing_simulated_EEG_observable_contract,"No same-unit, same-channel, same-preprocessing...",hard_gate
5,A_baseline14,ibi_cv,missing_simulated_EEG_observable_contract,"No same-unit, same-channel, same-preprocessing...",hard_gate
6,A_baseline14,pac_up_down_ratio,missing_simulated_EEG_observable_contract,"No same-unit, same-channel, same-preprocessing...",hard_gate
7,A_baseline14,spindle_density_per_min,missing_simulated_EEG_observable_contract,"No same-unit, same-channel, same-preprocessing...",hard_gate
8,A_baseline14,spindle_mean_duration_s,missing_simulated_EEG_observable_contract,"No same-unit, same-channel, same-preprocessing...",hard_gate
9,A_baseline14,pac_mi,missing_simulated_EEG_observable_contract,"No same-unit, same-channel, same-preprocessing...",hard_gate


,source,needle,line,excerpt
0,S4_sbi/simulator_wrapper.py,maps 4D theta,4,SBI simulator: maps 4D theta -> len(SUMMARY_KE...
1,S4_sbi/simulator_wrapper.py,SUMMARY_KEYS =,88,SUMMARY_KEYS = [
2,S4_sbi/simulator_wrapper.py,firing-rate key,105,"_EXC = v7.EXC # ""EXC"" firing-rate key"
3,S4_sbi/simulator_wrapper.py,Extract cortex (index 0) and thalamus (index 1...,204,# Extract cortex (index 0) and thalamus (index...
4,S4_sbi/simulator_wrapper.py,return np.array([stats[k] for k in SUMMARY_KEYS],223,return np.array([stats[k] for k in SUMMARY_KEY...
5,S4_sbi/src/sleep_sbi/observation.py,def _read_manifest,122,def _read_manifest(path: Path) -> pd.DataFrame:
6,S4_sbi/src/sleep_sbi/observation.py,def _pick_channel,157,"def _pick_channel(raw: mne.io.BaseRaw, candida..."
7,S4_sbi/src/sleep_sbi/observation.py,max_peak_to_peak_uv,71,"max_peak_to_peak_uv=float(payload[""qc""][""max_p..."
8,S4_sbi/src/sleep_sbi/observation.py,def build_observation_bundle,1125,def build_observation_bundle(


,label,path,exists,size_bytes,usable_as_matched_observation_bank,reason,keys,array_shapes
0,V7 fitted parameter JSON,data/patient_params_fig7_v7_SC4001.json,True,921,False,Candidate parameters or rate diagnostics lack ...,NaN,NaN
1,V8 fitted parameter JSON,data/patient_params_fig7_v8_SC4001.json,True,1146,False,Candidate parameters or rate diagnostics lack ...,NaN,NaN
2,V7 Pareto seed JSON,S4_v7_repair/pareto_seeds_fresh_DE.json,True,2383,False,Candidate parameters or rate diagnostics lack ...,NaN,NaN
3,V8a local-search best JSON,outputs/v8a_ultra_narrow_t6_t13_search/best_so...,True,877,False,Candidate parameters or rate diagnostics lack ...,NaN,NaN
4,V8a candidate archive,outputs/figure10_inspired_validation_panel/can...,True,13852,False,Candidate parameters or rate diagnostics lack ...,NaN,NaN
5,V7 raw rate diagnostic,outputs/v7_phase_diagnosis_signals.npz,True,2850298,False,Candidate parameters or rate diagnostics lack ...,"[""r_ctx"", ""r_thal"", ""so_filt"", ""so_phase"", ""sp...","{""r_ctx"": [60000], ""r_thal"": [60000], ""so_filt..."


,candidate_id,label,source,selected_for_simulation,simulation_status,gate_reason,mue,mui,b,tauA,g_LK,g_h,c_th2ctx,c_ctx2th
0,v7_fitted,V7 fitted parameter JSON,data/patient_params_fig7_v7_SC4001.json,False,not simulated,Extractor parity is NO-GO: candidate parameter...,3.801329,2.724747,42.548128,1042.897340,0.052266,0.066072,0.012676,0.056227
1,v8_fitted,V8 fitted parameter JSON,data/patient_params_fig7_v8_SC4001.json,False,not simulated,Extractor parity is NO-GO: candidate parameter...,4.151310,2.719276,32.472799,1418.554270,0.051219,0.053043,0.029332,0.131831
2,v8a_local_best,V8a/T13 local-search best JSON,outputs/v8a_ultra_narrow_t6_t13_search/best_so...,False,not simulated,Extractor parity is NO-GO: candidate parameter...,3.521318,2.885064,35.851259,1220.029899,0.050246,0.054287,0.067380,0.125349
3,v7_pareto_A,V7 Pareto seed A,S4_v7_repair/pareto_seeds_fresh_DE.json,False,not simulated,Extractor parity is NO-GO: candidate parameter...,3.473090,3.447265,34.408730,1229.516668,0.055812,0.058193,0.027498,0.059019
4,v7_pareto_B,V7 Pareto seed B,S4_v7_repair/pareto_seeds_fresh_DE.json,False,not simulated,Extractor parity is NO-GO: candidate parameter...,3.340686,3.275827,41.839010,1257.409182,0.052357,0.055031,0.032953,0.099784
5,v7_pareto_C,V7 Pareto seed C,S4_v7_repair/pareto_seeds_fresh_DE.json,False,not simulated,Extractor parity is NO-GO: candidate parameter...,3.448031,3.120446,36.512514,1825.816578,0.049741,0.063521,0.018107,0.122989
6,cand_0000,Candidate archive top-row inventory,outputs/figure10_inspired_validation_panel/can...,False,not simulated,Extractor parity is NO-GO: candidate parameter...,2.554340,4.638377,39.765021,4305.440791,0.051188,0.153022,0.099286,0.058737
7,cand_0001,Candidate archive top-row inventory,outputs/figure10_inspired_validation_panel/can...,False,not simulated,Extractor parity is NO-GO: candidate parameter...,4.021153,3.420452,46.433252,749.614315,0.028865,0.033163,0.142368,0.262521
8,cand_0002,Candidate archive top-row inventory,outputs/figure10_inspired_validation_panel/can...,False,not simulated,Extractor parity is NO-GO: candidate parameter...,2.558203,2.557749,25.495374,1725.761205,0.088110,0.072320,0.129517,0.016756


## Hard gate result

只有 fixed order、fixed length、finite/validity handling、same semantics、frozen aggregation/scaling 和 held-out allocation 都明确的 schema 才能进入 05。No-Go 是诚实的 scientific result。

In [4]:
manifest = parity_result["manifest"]
print(json.dumps(manifest["hard_gate"], indent=2))
assert manifest["hard_gate"]["status"] == "NO-GO"
assert not manifest["hard_gate"]["passed_schemas"]
assert not parity_result["parity"]["parity_pass"].any()

{
  "passed_schemas": [],
  "status": "NO-GO",
  "reason": "No fixed-length, same-semantic simulated EEG observable schema exists. Existing banks are legacy 5D/7D rate-summary banks and cannot be relabelled."
}
